In [6]:
# ── LSTM.ipynb — imports, reproducibility, device ────────────────────────
import numpy as np
import pandas as pd
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

import features as F

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)          # 'mps' on Apple Silicon — much faster than cpu

SEQ_LEN  = 55
VAL_FRAC = 0.2

device: mps


## Data Loading

In [7]:
# ── Load + engineer features (shared module) ─────────────────────────────
df = pd.read_csv("Data/train.csv")
df = F.build_features(df)                 # sorted (stock, date, seconds) + features
feat_cols = F.feature_columns(df)
print(f"{len(feat_cols)} per-timestep features")

33 per-timestep features


In [8]:
# ── Load + engineer features (shared module) ─────────────────────────────
df = pd.read_csv("Data/train.csv")
df = F.build_features(df)                 # sorted (stock, date, seconds) + features
feat_cols = F.feature_columns(df)
print(f"{len(feat_cols)} per-timestep features")

33 per-timestep features


In [9]:
# ── Scale (fit on TRAIN dates only) + clean NaN/inf ──────────────────────
dates  = np.sort(df["date_id"].unique())
cutoff = dates[-int(len(dates) * VAL_FRAC)]        # first validation date

X_all    = df[feat_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)
scaler   = StandardScaler().fit(X_all[df["date_id"] < cutoff])
X_scaled = scaler.transform(X_all).astype(np.float32)

In [10]:
# ── Reshape flat rows -> (n_seq, 55, n_feat) sequences ──────────────────
counts = df.groupby(["stock_id", "date_id"]).size()
assert (counts == SEQ_LEN).all(), "a stock-day isn't 55 steps long"
n_seq = len(counts)

X_seq    = X_scaled.reshape(n_seq, SEQ_LEN, len(feat_cols))
y_seq    = df["target"].to_numpy(np.float32).reshape(n_seq, SEQ_LEN)
mask_seq = (~np.isnan(y_seq)).astype(np.float32)     # 1=valid target, 0=NaN
y_seq    = np.nan_to_num(y_seq, nan=0.0)

first_rows = df.iloc[::SEQ_LEN]                       # first step of each stock-day
seq_date   = first_rows["date_id"].to_numpy()
stock_ids  = np.sort(df["stock_id"].unique())
stock_to_idx  = {s: i for i, s in enumerate(stock_ids)}
seq_stock_idx = first_rows["stock_id"].map(stock_to_idx).to_numpy(np.int64)

In [11]:
# ── Split sequences by date, wrap in tensors + loaders ──────────────────
tr = seq_date < cutoff
def to_tensors(idx):
    return (torch.tensor(X_seq[idx]),       torch.tensor(seq_stock_idx[idx]),
            torch.tensor(y_seq[idx]),       torch.tensor(mask_seq[idx]))

Xtr, Str, ytr, mtr = to_tensors(tr)
Xva, Sva, yva, mva = to_tensors(~tr)

train_loader = DataLoader(TensorDataset(Xtr, Str, ytr, mtr), batch_size=256, shuffle=True)
val_loader   = DataLoader(TensorDataset(Xva, Sva, yva, mva), batch_size=512, shuffle=False)
n_features, n_stocks = len(feat_cols), len(stock_ids)
print(f"train {tr.sum():,} | val {(~tr).sum():,} seqs | X {tuple(Xtr.shape)} | "
      f"{n_features} feats, {n_stocks} stocks")

train 76,036 | val 19,200 seqs | X (76036, 55, 33) | 33 feats, 200 stocks


## Modellimng

In [12]:
g = pd.read_parquet("preds_gru_val.parquet")
l = pd.read_parquet("preds_lgb_val.parquet")
m = g.merge(l, on="row_id").dropna(subset=["target"])

# THE headline diagnostic: how decorrelated are the two models?
print(f"corr(gru, lgb) = {m['gru'].corr(m['lgb']):.3f}")

zero = m["target"].abs().mean()
def skill(p): return (zero - (m["target"] - p).abs().mean()) / zero * 100

# search the blend weight (w on lgb, 1-w on gru)
for w in [0.0, 0.3, 0.4, 0.5, 0.6, 0.7, 1.0]:
    blend = w * m["lgb"] + (1 - w) * m["gru"]
    # zero-sum post-processing on the blend
    blend_adj = blend - blend.groupby([m.date_id, m.seconds_in_bucket]).transform("mean")
    print(f"w_lgb={w:.1f}  blend={skill(blend):.2f}%  +zerosum={skill(blend_adj):.2f}%")

corr(gru, lgb) = 0.835
w_lgb=0.0  blend=2.07%  +zerosum=2.10%
w_lgb=0.3  blend=2.22%  +zerosum=2.24%
w_lgb=0.4  blend=2.24%  +zerosum=2.26%
w_lgb=0.5  blend=2.24%  +zerosum=2.27%
w_lgb=0.6  blend=2.23%  +zerosum=2.26%
w_lgb=0.7  blend=2.20%  +zerosum=2.23%
w_lgb=1.0  blend=2.02%  +zerosum=2.06%


In [14]:
# ── Model: stock embedding + unidirectional RNN -> per-timestep output ────
import torch.nn as nn

class SeqModel(nn.Module):
    """GRU or LSTM over the 55-step auction sequence. Predicts a target at
    EVERY timestep. Unidirectional by design — see the note below."""
    def __init__(self, n_features, n_stocks, rnn="gru",
                 emb_dim=24, hidden=128, layers=2, dropout=0.2):
        super().__init__()
        self.emb = nn.Embedding(n_stocks, emb_dim)
        rnn_cls = nn.GRU if rnn == "gru" else nn.LSTM
        self.rnn = rnn_cls(
            input_size=n_features + emb_dim,
            hidden_size=hidden, num_layers=layers,
            batch_first=True, dropout=dropout,
            bidirectional=False,        # <-- CRITICAL: never True (see note)
        )
        self.head = nn.Sequential(nn.Linear(hidden, 64), nn.ReLU(), nn.Linear(64, 1))

    def forward(self, x, stock):                     # x: (B,55,F)  stock: (B,)
        e = self.emb(stock).unsqueeze(1).expand(-1, x.size(1), -1)  # (B,55,emb)
        out, _ = self.rnn(torch.cat([x, e], dim=-1))               # (B,55,hidden)
        return self.head(out).squeeze(-1)                          # (B,55)

In [16]:
# ── Masked MAE: the 88 NaN-target steps must contribute zero loss ────────
def masked_mae(pred, y, mask):
    return (torch.abs(pred - y) * mask).sum() / mask.sum()

@torch.no_grad()
def eval_mae(model, loader):
    """Exact masked MAE over a loader (matches the competition metric)."""
    model.eval()
    err, n = 0.0, 0.0
    for x, s, y, m in loader:
        x, s, y, m = x.to(device), s.to(device), y.to(device), m.to(device)
        p = model(x, s)
        err += (torch.abs(p - y) * m).sum().item()
        n   += m.sum().item()
    return err / n

In [17]:
# ── Confirm the GRU+LGB blend across walk-forward folds ──────────────────
import lightgbm as lgb

def _loaders(tr_seq_mask, va_seq_mask):
    def ds(mask):
        idx = np.where(mask)[0]
        return TensorDataset(torch.tensor(X_seq[idx]), torch.tensor(seq_stock_idx[idx]),
                             torch.tensor(y_seq[idx]), torch.tensor(mask_seq[idx]))
    return (DataLoader(ds(tr_seq_mask), batch_size=256, shuffle=True),
            DataLoader(ds(va_seq_mask), batch_size=512, shuffle=False))

def _train_gru(tl, vl, va_idx, seed=0, epochs=40):
    torch.manual_seed(seed); np.random.seed(seed)
    model = SeqModel(n_features, n_stocks, "gru").to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    best, best_state, bad = 1e9, None, 0
    for _ in range(epochs):
        model.train()
        for x, s, y, mk in tl:
            x, s, y, mk = x.to(device), s.to(device), y.to(device), mk.to(device)
            opt.zero_grad(); masked_mae(model(x, s), y, mk).backward(); opt.step()
        v = eval_mae(model, vl)
        if v < best - 1e-4:
            best, best_state, bad = v, {k: t.cpu().clone() for k, t in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= 4: break
    model.load_state_dict(best_state); model.eval()
    with torch.no_grad():
        p = model(torch.tensor(X_seq[va_idx]).to(device),
                  torch.tensor(seq_stock_idx[va_idx]).to(device)).cpu().numpy()
    return p.reshape(-1)                                   # aligned to df val rows

def blend_cv(lgb_params, n_folds=3, val_days=50, w_lgb=0.5, seed=0):
    all_dates = np.sort(df.date_id.unique())
    rows = []
    for k in range(n_folds):
        ve = len(all_dates) - (n_folds - 1 - k) * val_days
        val_dates, tr_dates = all_dates[ve - val_days:ve], all_dates[:ve - val_days]
        va_idx  = np.where(np.isin(seq_date, val_dates))[0]
        m_tr, m_va = df.date_id.isin(tr_dates).values, df.date_id.isin(val_dates).values
        dval = df[m_va]

        # LightGBM (flat rows; shared features.py set + stock_id)
        cols = feat_cols + ["stock_id"]
        Xtr = df.loc[m_tr, cols].replace([np.inf, -np.inf], np.nan).fillna(0)
        ytr = df.loc[m_tr, "target"]; ok = ytr.notna().values
        Xva = df.loc[m_va, cols].replace([np.inf, -np.inf], np.nan).fillna(0)
        gbm = lgb.LGBMRegressor(**lgb_params)
        gbm.fit(Xtr[ok], ytr[ok], categorical_feature=["stock_id"])
        p_lgb = gbm.predict(Xva)

        # GRU (sequences)
        tl, vl = _loaders(np.isin(seq_date, tr_dates), np.isin(seq_date, val_dates))
        p_gru = _train_gru(tl, vl, va_idx, seed=seed)

        # align (dval order == p_lgb == p_gru) + score
        y = dval.target.values; ok_v = ~np.isnan(y)
        zero = np.abs(y[ok_v]).mean()
        sk = lambda p: (zero - np.abs(y[ok_v] - p[ok_v]).mean()) / zero * 100
        blend = w_lgb * p_lgb + (1 - w_lgb) * p_gru
        key = (dval.date_id.astype(str) + "_" + dval.seconds_in_bucket.astype(str)).values
        bser = pd.Series(blend)
        blend_zs = (bser - bser.groupby(key).transform("mean")).values
        r = dict(fold=k, lgb=sk(p_lgb), gru=sk(p_gru), blend=sk(blend),
                 blend_zs=sk(blend_zs), corr=np.corrcoef(p_lgb[ok_v], p_gru[ok_v])[0, 1])
        print(f"fold {k}: lgb={r['lgb']:.2f}  gru={r['gru']:.2f}  blend={r['blend']:.2f}  "
              f"blend+zs={r['blend_zs']:.2f}  corr={r['corr']:.3f}")
        rows.append(r)
    res = pd.DataFrame(rows)
    print(f"\nMEAN  lgb={res.lgb.mean():.2f}  gru={res.gru.mean():.2f}  "
          f"blend={res.blend.mean():.2f}  blend+zs={res.blend_zs.mean():.2f}")
    return res

lgb_params = dict(objective="mae", n_estimators=500, learning_rate=0.03, num_leaves=31,
                  min_child_samples=1000, reg_alpha=1.0, reg_lambda=1.0,
                  subsample=0.7, subsample_freq=1, colsample_bytree=0.7,
                  random_state=42, n_jobs=-1, verbose=-1)

cv = blend_cv(lgb_params, n_folds=3, val_days=50)

fold 0: lgb=1.81  gru=1.91  blend=2.12  blend+zs=2.09  corr=0.798
fold 1: lgb=1.86  gru=2.00  blend=2.17  blend+zs=2.17  corr=0.816
fold 2: lgb=1.68  gru=1.93  blend=2.04  blend+zs=2.08  corr=0.791

MEAN  lgb=1.78  gru=1.95  blend=2.11  blend+zs=2.12
